In [3]:
from pyspark.sql import SparkSession
import mysql.connector
from pyspark.sql.functions import *

# Step 1: Create Spark Session
spark = SparkSession.builder \
    .appName("MySQL JDBC Example") \
    .config("spark.jars", "/Users/mugesh_krishna/Downloads/mysql-connector-j-9.3.0/mysql-connector-j-9.3.0.jar") \
    .getOrCreate()

print(spark)

print("Spark Session created!")

jdbc_url = "jdbc:mysql://localhost:3306/samplebill"
properties = {
    "user": "root",
    "password": "mugesh585",
    "driver": "com.mysql.cj.jdbc.Driver"
}


df = spark.read.jdbc(url=jdbc_url, table="billinfo", properties=properties)
print("All rows from billinfo table:")
df.schema
df.show()
print(df["name"])
sample=df.select(df.name).where(col("billdate")=="2025-06-10")
sample.show()
sample.write.format("csv").save("data write/sample.csv")
sp=sample.select(upper(df.name).alias("CAPS NAME"))
sp.show()
filtered_df = df.filter((df["ispackage"] == "No") | (df["ispackage"] == "no"))
id_list = [row["billid"] for row in filtered_df.select("billid").collect()]
print("IDs with ispackage = 'No':", id_list)

if id_list:
    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password="mugesh585",
            database="samplebill"
        )
        cursor = conn.cursor()

        format_ids = ','.join(['%s'] * len(id_list))
        update_query = f"UPDATE billinfo SET ispackage = 'Yes' WHERE billid IN ({format_ids})"
        cursor.execute(update_query, id_list)
        conn.commit()

        print(f"Updated {cursor.rowcount} rows where ispackage was 'No'.")
    except Exception as e:
        print("Error updating MySQL:", e)
    finally:
        cursor.close()
        conn.close()
else:
    print("No records found where ispackage = 'No'.")

 


repli_df = spark.read.jdbc(url=jdbc_url, table="billrepli", properties=properties)

repli_ids_df = repli_df.select("billid").distinct()


new_records_df = df.join(repli_ids_df, on="billid", how="left_anti")

print("New records to replicate from billinfo to billrepli:")
new_records_df.show()


new_records_to_insert = new_records_df.select(
    "billid","name", "origin", "desti", "billdate", "deliverydate", "ispackage"
)

new_records_to_insert.write.jdbc(
    url=jdbc_url,
    table="billrepli",
    mode="append",
    properties=properties
)
print(f"{new_records_to_insert.count()} new record(s) replicated to billrepli.")


Spark Session created!
All rows from billinfo table:
+------+-------------+----------+-----------+----------+------------+---------+
|billid|         name|    origin|      desti|  billdate|deliverydate|ispackage|
+------+-------------+----------+-----------+----------+------------+---------+
|   101|     Gopinath|   Chennai|    Denmark|2025-06-03|  2025-07-10|      Yes|
|   102|     Karthick|    Mumbai|  Singapore|2025-06-03|  2025-07-15|      Yes|
|   103|Krishna Kumar|     Dubai|    Hamburg|2025-06-03|  2025-08-05|      Yes|
|   104|      Gowshik|   Denmark|    Denmark|2025-06-03|  2025-09-20|      Yes|
|   105|       Dinesh| Singapore|    Chennai|2025-06-04|  2025-09-12|      Yes|
|   106|        Muthu|     China|     Mumbai|2025-06-04|  2025-09-20|      Yes|
|   107|      Agila K| Rotterdam|     Mumbai|2025-06-10|  2025-09-20|      Yes|
|   108|         Raju|   Denmark|   Bulgaria|2025-06-10|  2025-10-20|      Yes|
|   109|    Kumaravel|    Mexico|Switzerland|2025-06-10|  2025-09-1

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/Users/mugesh_krishna/data write/sample.csv already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

In [2]:
import mysql.connector
from datetime import datetime


try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="mugesh585",
        database="samplebill"
    )

    cursor = conn.cursor()
    print("Connected to MySQL database")

    # Getting inputs from user
    billid = int(input("Enter Bill ID (int): "))
    name = input("Enter Name: ")
    origin = input("Enter Origin: ")
    desti = input("Enter Destination: ")
    billdate = input("Enter Bill Date (YYYY-MM-DD): ")
    deliverydate = input("Enter Delivery Date (YYYY-MM-DD): ")
    ispackage = input("Is it a package? (Yes/No): ")

   
    insert_query = """
    INSERT INTO billinfo (billid, name, origin, desti, billdate, deliverydate, ispackage)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """

   
    cursor.execute(insert_query, (billid, name, origin, desti, billdate, deliverydate, ispackage))
    conn.commit()

    print(" Data inserted successfully into 'billinfo' table.")

except mysql.connector.Error as err:
    print(" Error:", err)

finally:
    if conn.is_connected():
        cursor.close()
        conn.close()
        print(" MySQL connection closed.")


Connected to MySQL database


Enter Bill ID (int):  119
Enter Name:  Nithish
Enter Origin:  Denmark
Enter Destination:  Cape Town
Enter Bill Date (YYYY-MM-DD):  2025-06-16
Enter Delivery Date (YYYY-MM-DD):  2025-09-10
Is it a package? (Yes/No):  Yes


 Data inserted successfully into 'billinfo' table.
 MySQL connection closed.


In [3]:
pip install pytz

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
import mysql.connector

# Step 1: Get current IST time
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)
hour = current_time.hour
minute = current_time.minute


if hour == 15 and minute == 0:
    # Step 3: Create Spark session
    spark = SparkSession.builder \
        .appName("Scheduled Replication Task") \
        .config("spark.jars", "/Users/mugesh_krishna/Downloads/mysql-connector-j-9.3.0/mysql-connector-j-9.3.0.jar") \
        .getOrCreate()

    print("Starting replication at 15:00 IST...")

    # JDBC connection info
    jdbc_url = "jdbc:mysql://localhost:3306/samplebill"
    properties = {
        "user": "root",
        "password": "mugesh585",
        "driver": "com.mysql.cj.jdbc.Driver"
    }

    # Step 4: Read source and target tables
    df = spark.read.jdbc(url=jdbc_url, table="billinfo", properties=properties)
    repli_df = spark.read.jdbc(url=jdbc_url, table="billrepli", properties=properties)

    # Step 5: Find new records not in billrepli
    repli_ids_df = repli_df.select("billid").distinct()
    new_records_df = df.join(repli_ids_df, on="billid", how="left_anti")

    # Step 6: Insert only new records
    if new_records_df.count() > 0:
        new_records_df.select(
            "billid", "name", "origin", "desti", "billdate", "deliverydate", "ispackage"
        ).write.jdbc(
            url=jdbc_url,
            table="billrepli",
            mode="append",
            properties=properties
        )
        print(f"{new_records_df.count()} new record(s) replicated to billrepli.")
    else:
        print("No new records to replicate.")
    spark.stop()
else:
    print(f"Current IST time is {current_time.strftime('%H:%M')}. Not 15:00 yet, skipping replication.")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/06/10 12:00:11 WARN Utils: Your hostname, Mugesh-Krishna.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.53 instead (on interface en0)
25/06/10 12:00:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/06/10 12:00:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Starting replication at 15:00 IST...
0 new record(s) replicated to billrepli.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Start Spark
spark = SparkSession.builder \
    .appName("MySQL Replication with Update on Change") \
    .config("spark.jars", "/Users/mugesh_krishna/Downloads/mysql-connector-j-9.3.0/mysql-connector-j-9.3.0.jar") \
    .getOrCreate()

jdbc_url = "jdbc:mysql://localhost:3306/samplebill"
properties = {
    "user": "root",
    "password": "mugesh585",
    "driver": "com.mysql.cj.jdbc.Driver"
}

# 2. Load source and target tables
billinfo_df = spark.read.jdbc(url=jdbc_url, table="billinfo", properties=properties)
billrepli_df = spark.read.jdbc(url=jdbc_url, table="billrepli", properties=properties)

# 3. Join on primary key (billid)
joined_df = billinfo_df.alias("src").join(
    billrepli_df.alias("tgt"), on="billid", how="outer"
)

# 4. Identify records that are new or changed
changed_df = joined_df.filter(
    (col("tgt.billid").isNull()) |  # new record
    (col("src.name") != col("tgt.name")) |
    (col("src.origin") != col("tgt.origin")) |
    (col("src.desti") != col("tgt.desti")) |
    (col("src.billdate") != col("tgt.billdate")) |
    (col("src.deliverydate") != col("tgt.deliverydate")) |
    (col("src.ispackage") != col("tgt.ispackage"))
).select(
    "src.billid", "src.name", "src.origin", "src.desti", "src.billdate", "src.deliverydate", "src.ispackage"
)

# 5. Write updated records back to billrepli
# Note: Replace/merge logic requires either:
# - temp staging table + manual update
# - OR truncate and reinsert all changed (if safe)

# Option 1: Upsert using overwrite for changed rows only
if changed_df.count() > 0:
    changed_df.write.jdbc(
        url=jdbc_url,
        table="billrepli_staging",
        mode="overwrite",
        properties=properties
    )

    print(f"{changed_df.count()} records written to staging table.")

    # Optional: You can now use SQL (from Python or manually) to MERGE/UPSERT from `billrepli_staging` to `billrepli`
    # via `ON DUPLICATE KEY UPDATE` in MySQL

    # Example query (run with cursor.execute if using mysql.connector)
    # """
    # INSERT INTO billrepli (billid, name, origin, desti, billdate, deliverydate, ispackage)
    # SELECT billid, name, origin, desti, billdate, deliverydate, ispackage FROM billrepli_staging
    # ON DUPLICATE KEY UPDATE 
    #     name=VALUES(name), origin=VALUES(origin), desti=VALUES(desti), 
    #     billdate=VALUES(billdate), deliverydate=VALUES(deliverydate), ispackage=VALUES(ispackage);
    # """

else:
    print("No new or changed records found.")


25/06/12 17:53:03 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


1 records written to staging table.


In [5]:
import mysql.connector

try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="mugesh585",
        database="samplebill"
    )
    cursor = conn.cursor()

    merge_sql = """
        INSERT INTO billrepli (billid, name, origin, desti, billdate, deliverydate, ispackage)
        SELECT billid, name, origin, desti, billdate, deliverydate, ispackage FROM billrepli_staging
        ON DUPLICATE KEY UPDATE
            name = VALUES(name),
            origin = VALUES(origin),
            desti = VALUES(desti),
            billdate = VALUES(billdate),
            deliverydate = VALUES(deliverydate),
            ispackage = VALUES(ispackage);
    """

    cursor.execute(merge_sql)
    conn.commit()
    print(f"{cursor.rowcount} rows merged into billrepli.")

except mysql.connector.Error as err:
    print("MySQL Error:", err)

finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()


1 rows merged into billrepli.


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper
import mysql.connector

# Step 1: Start Spark Session
spark = SparkSession.builder \
    .appName("MySQL Replication with Update on Change") \
    .config("spark.jars", "/Users/mugesh_krishna/Downloads/mysql-connector-j-9.3.0/mysql-connector-j-9.3.0.jar") \
    .getOrCreate()

print("Spark Session created!")

# Step 2: Define JDBC details
jdbc_url = "jdbc:mysql://localhost:3306/samplebill"
properties = {
    "user": "root",
    "password": "mugesh585",
    "driver": "com.mysql.cj.jdbc.Driver"
}


billinfo_df = spark.read.jdbc(url=jdbc_url, table="billinfo", properties=properties)
billrepli_df = spark.read.jdbc(url=jdbc_url, table="billrepli", properties=properties)

# seeing which  are new or changed in my table 
joined_df = billinfo_df.alias("src").join(
    billrepli_df.alias("tgt"), on="billid", how="outer"
)

changed_df = joined_df.filter(
    (col("tgt.billid").isNull()) |
    (col("src.name") != col("tgt.name")) |
    (col("src.origin") != col("tgt.origin")) |
    (col("src.desti") != col("tgt.desti")) |
    (col("src.billdate") != col("tgt.billdate")) |
    (col("src.deliverydate") != col("tgt.deliverydate")) |
    (col("src.ispackage") != col("tgt.ispackage"))
).select(
    "src.billid", "src.name", "src.origin", "src.desti", "src.billdate", "src.deliverydate", "src.ispackage"
)

# Step 5: Write changed records to staging table
if changed_df.count() > 0:
    changed_df.write.jdbc(
        url=jdbc_url,
        table="billrepli_staging",
        mode="overwrite",
        properties=properties
    )
    print(f"{changed_df.count()} records written to staging table.")
else:
    print("No new or changed records found.")

# Step 6: Update records in billrepli from billrepli_staging using merge
try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="mugesh585",
        database="samplebill"
    )
    cursor = conn.cursor()

    merge_sql = """
        INSERT INTO billrepli (billid, name, origin, desti, billdate, deliverydate, ispackage)
        SELECT billid, name, origin, desti, billdate, deliverydate, ispackage FROM billrepli_staging
        ON DUPLICATE KEY UPDATE
            name = VALUES(name),
            origin = VALUES(origin),
            desti = VALUES(desti),
            billdate = VALUES(billdate),
            deliverydate = VALUES(deliverydate),
            ispackage = VALUES(ispackage);
    """
    cursor.execute(merge_sql)
    conn.commit()
    print(f"{cursor.rowcount} rows merged into billrepli.")

except mysql.connector.Error as err:
    print("MySQL Error:", err)

finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()

# Step 7: Replicate brand-new records from billinfo to billrepli
repli_ids_df = billrepli_df.select("billid").distinct()
new_records_df = billinfo_df.join(repli_ids_df, on="billid", how="left_anti")

if new_records_df.count() > 0:
    new_records_df.select(
        "billid", "name", "origin", "desti", "billdate", "deliverydate", "ispackage"
    ).write.jdbc(
        url=jdbc_url,
        table="billrepli",
        mode="append",
        properties=properties
    )
    print(f"{new_records_df.count()} new record(s) replicated to billrepli.")
else:
    print("No new records to replicate.")

# Step 8: Mark 'ispackage' = 'No' records in billinfo as 'Yes'
filtered_df = billinfo_df.filter((col("ispackage") == "No") | (col("ispackage") == "no"))
id_list = [row["billid"] for row in filtered_df.select("billid").collect()]

if id_list:
    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password="mugesh585",
            database="samplebill"
        )
        cursor = conn.cursor()

        format_ids = ','.join(['%s'] * len(id_list))
        update_query = f"UPDATE billinfo SET ispackage = 'Yes' WHERE billid IN ({format_ids})"
        cursor.execute(update_query, id_list)
        conn.commit()

        print(f"Updated {cursor.rowcount} rows where ispackage was 'No'.")

    except Exception as e:
        print("Error updating MySQL:", e)
    finally:
        cursor.close()
        conn.close()
else:
    print("No records found where ispackage = 'No'.")

Spark Session created!
No new or changed records found.
0 rows merged into billrepli.
No new records to replicate.
No records found where ispackage = 'No'.
